In [5]:
# 1. Setup and Imports

import pymupdf as pdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Paths
pdf_path = "../data/Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf"
document_output_path = "gbr_extracted_text.txt"

# Page selection: pages 6 – 86 
selected_pages = list(range(5, 87))

# Defining Metadata for the document
metadata = {
    "source": "Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf",
    "author": "Great Barrier Reef Marine Park Authority",
    "year": 2007,
    "description": "A comprehensive study on the economic contribution of the Great Barrier Reef Marine Park for the years 2005-2006.",
}


In [6]:
# 2. Extracting the text from the PDF
def extract_text_from_pdf(pdf_path: str, selected_pages: list[int], output_file: str) -> None:
    doc = pdf.open(pdf_path)
    with open(output_file, "wb") as out:
        for page_num in selected_pages:
            if page_num < len(doc):
                page = doc[page_num]
                text = page.get_text()
                text = " ".join(text.split()) 
                out.write(text.encode("utf8"))
                out.write(b"\n\n")
    doc.close()

    print(f"Extracted {len(selected_pages)} pages to {output_file}")

In [7]:
# 3. Execute PDF extraction

extract_text_from_pdf(pdf_path, selected_pages, document_output_path)

print(f"PDF extraction complete")

Extracted 82 pages to gbr_extracted_text.txt
PDF extraction complete


In [8]:
# 4. Load text file with previously defined metadata

loader = TextLoader(document_output_path, encoding="utf-8")
docs = loader.load()

# Add metadata to documents
for doc in docs:
    doc.metadata.update(metadata)



In [9]:
# 5. Data cleaning: white space removal
# Boilerplate removal: all emails, urls, page numbers, copyright info, PDF generation info

import re

def clean_text_for_bge(text: str) -> str:
    
    text = re.sub(r'[^\x00-\x7F\u00A0-\uFFFF]', ' ', text)
    
    text = re.sub(r'http[s]?://[^\s]+|www\.[^\s]+', '', text)
    text = re.sub(r'page?\s*\d+\s*(of\s*)?\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'seite?\s*\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'©?\s*(?:great barrier reef|marine park|\d{4})', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    text = re.sub(r'all rights reserved', '', text, flags=re.IGNORECASE)
    text = re.sub(r'generated by adobe|document id|created with', '', text, flags=re.IGNORECASE)
    
    text = " ".join(text.split())
    
    return text.strip()

for doc in docs:
    doc.page_content = clean_text_for_bge(doc.page_content)


test = "Text... © Great Barrier Reef Marine Park Authority Page 5 of 10"
print(clean_text_for_bge(test))



Text... Authority


In [10]:
# 6. Data exploration: analyze text before chunking (how often does one word appear, how long are the documents, any outliers?)

from collections import Counter
import statistics
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

# Basic statistics
total_chars = sum(len(doc.page_content) for doc in docs)
total_words = sum(len(doc.page_content.split()) for doc in docs)
doc_lengths = [len(doc.page_content) for doc in docs]

print("=== Data Overview ===")
print(f"Total documents: {len(docs)}")
print(f"Total characters: {total_chars:,}")
print(f"Total words: {total_words:,}")
print(f"Avg doc length: {statistics.mean(doc_lengths):,.0f} chars")
print(f"Min/Max doc length: {min(doc_lengths):,} / {max(doc_lengths):,} chars")

# Extract meaningful keywords (filtered)
print("\n=== Top 10 Keywords (filtered) ===")
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

# Filter: only alphabetic characters, length > 2, and not stopwords
# check later if words > 2 was a good threshold
all_words = [w for w in " ".join(doc.page_content.lower() for doc in docs).split() 
             if w.isalpha() and len(w) > 2]
keywords = [w for w in all_words if w not in stop_words]

keyword_freq = Counter(keywords)
for word, count in keyword_freq.most_common(10):
    print(f"  {word}: {count}")

# Check for very short documents (potential outliers)
print("\n=== Potential Issues ===")
short_docs = [d for d in docs if len(d.page_content) < 100]
if short_docs:
    print(f"⚠ Found {len(short_docs)} very short documents (< 100 chars)")
    print(f"  First example: '{short_docs[0].page_content}'")
else:
    print("No unusually short documents")

=== Data Overview ===
Total documents: 1
Total characters: 211,654
Total words: 33,338
Avg doc length: 211,654 chars
Min/Max doc length: 211,654 / 211,654 chars

=== Top 10 Keywords (filtered) ===
  gbrca: 344
  tourism: 283
  total: 194
  visitors: 172
  value: 168
  recreational: 167
  fishing: 167
  queensland: 157
  data: 145
  commercial: 138

=== Potential Issues ===
No unusually short documents


In [ ]:
# 7. Parsing and chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(f"  Content: {chunk.page_content[:200]}...")  # First 200 chars
    print(f"  Metadata: {chunk.metadata}")
    print()



In [12]:
# 7.1. Remove duplicates in data if exist

def remove_duplicate_chunks(chunks):
    seen = set()
    unique_chunks = []
    
    for chunk in chunks:
        content_hash = hash(chunk.page_content)
        if content_hash not in seen:
            seen.add(content_hash)
            unique_chunks.append(chunk)
    
    print(f"Removed {len(chunks) - len(unique_chunks)} duplicates. {len(unique_chunks)} chunks remain.")
    return unique_chunks

chunks = remove_duplicate_chunks(chunks)

Removed 0 duplicates. 265 chunks remain.


In [13]:
# 8. Initialize embeddings and create vector store

import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# 9. Embedding class for BGE-M3

class EmbeddingManager:
    
    def __init__(self, model_name: str = "BAAI/bge-m3"):
        self.model_name = model_name
        self.model = None
        self.load_model()
    
    def load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model not loaded.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            return np.array(embeddings)
        except Exception as e:
            print(f"Error during embedding: {e}")
            raise

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Embedding model not loaded.")
        return self.model.get_sentence_embedding_dimension()
    

# Initialize embedding manager

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 65501.97it/s]


Loaded embedding model: BAAI/bge-m3. Embedding dimension: 1024
EmbeddingManager initialized successfully


In [15]:
# 10. Vector store class (FAISS)

import os
import faiss
import pickle 

class FaissVectorStore:

    def __init__(self, embedding_dim: int, persist_directory: str = "./data/vector_store"):
        self.embedding_dim = embedding_dim
        self.persist_directory = persist_directory
        self.index_path = os.path.join(persist_directory, "index.faiss")
        self.metadata_path = os.path.join(persist_directory, "metadata.pkl")
        
        # if exists skips the creation, otherwise creates the directory
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # load index if exists, otherwise create new
        if os.path.exists(self.index_path):
            self.index = faiss.read_index(self.index_path)
            with open(self.metadata_path, "rb") as f:
                self.id_to_metadata = pickle.load(f)
            print(f"Loaded index with{len(self.id_to_metadata)} Chunks")
        else:
            self.index = faiss.IndexFlatIP(embedding_dim) # IP = inner Product, equivalent to cosine similarity
            self.id_to_metadata = {}
            print("New index created.")

    def normalize_embeddings(self, embeddings):
            norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
            return embeddings / norms

    def add_embeddings(self, embeddings, metadatas):
        embeddings = self.normalize_embeddings(embeddings)
        self.index.add(embeddings.astype('float32'))
        
        # conncet metadata with IDs
        current_size = len(self.id_to_metadata)
        for i, meta in enumerate(metadatas):
            self.id_to_metadata[current_size + i] = meta
            
        # persistently save to disk
        faiss.write_index(self.index, self.index_path)
        with open(self.metadata_path, "wb") as f:
            pickle.dump(self.id_to_metadata, f)
        print(f"{len(metadatas)} Chunks safed to {self.persist_directory}")

    def search(self, query_embedding: np.ndarray, top_k: int = 5):
        query_embedding = self.normalize_embeddings(query_embedding.reshape(1, -1))
        distances, indices = self.index.search(query_embedding.astype('float32').reshape(1, -1), top_k)
        results = [(self.id_to_metadata[idx], distances[0][i]) for i, idx in enumerate(indices[0])]
        return results

In [16]:
# 10.2 Initialize vector store

vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
print("Vector store initialized successfully")

New index created.
Vector store initialized successfully


In [17]:
# 11. Check for duplicate chunks before embedding

# Initialization of vector store

def get_chunk_hash(chunk_content: str) -> str:
    import hashlib
    return hashlib.md5(chunk_content.encode()).hexdigest()

# Get hashes of new chunks
new_chunk_hashes = {get_chunk_hash(chunk.page_content): chunk for chunk in chunks}

# Get existing chunk hashes from vector store (what is already stored)
existing_hashes = set()
for metadata in vector_store.id_to_metadata.values():
    if "chunk_hash" in metadata:
        existing_hashes.add(metadata["chunk_hash"])

# Filter: only new chunks will be embedded to avoid double embeddings
chunks_to_add = [chunk for chunk_hash, chunk in new_chunk_hashes.items() 
                 if chunk_hash not in existing_hashes]

print(f"Total chunks: {len(chunks)}")
print(f"Already in vector store: {len(chunks) - len(chunks_to_add)}")
print(f"New chunks to add: {len(chunks_to_add)}")


Total chunks: 265
Already in vector store: 0
New chunks to add: 265


In [ ]:
#12. embed remaining chunks after removal of duplicates into created vecrotr store and save to disk
chunk_texts = [chunk.page_content for chunk in chunks_to_add]
embeddings = embedding_manager.generate_embeddings(chunk_texts) 
metadatas = [
    {
        "content": chunk.page_content, 
        "source": chunk.metadata,
        "chunk_hash": get_chunk_hash(chunk.page_content)
    } 
    for chunk in chunks_to_add
    ]
vector_store.add_embeddings(embeddings, metadatas)

print(f"Created {len(chunks_to_add)} chunks from {len(docs)} documents")
print(f"Chunk sizes: min={min(len(c.page_content) for c in chunks_to_add)}, max={max(len(c.page_content) for c in chunks_to_add)}")

Batches: 100%|██████████| 9/9 [00:27<00:00,  3.06s/it]

265 Chunks safed to ./data/vector_store
Created 265 chunks from 1 documents
Chunk sizes: min=453, max=999


NameError: name 'chunks_texts' is not defined

In [ ]:
#13. Repeat steps oin case that new documents are added 

